# Auto fix-plan store example

This notebook shows the read-only auto store. It exposes registered fixes as single-step fix plans, including fixes from plugins when those plugins are installed and loaded.

In [ ]:
import numpy as np

import woodpecker
from woodpecker.stores import AutoFixPlanStore
from woodpecker.testing import make_cmip6

Create a small CMIP6-like dataset where `tas` uses Celsius units. The core units fix can detect this from metadata.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

dataset

The auto store lists every registered fix as a one-step plan. The generated plan id is the fix id.

In [ ]:
store = AutoFixPlanStore()

[plan.id for plan in store.list_plans() if plan.id.startswith("woodpecker.")]

Lookup calls each fix implementation's own `matches()` method and returns only plans that apply to the dataset.

In [ ]:
matched_plans = store.lookup(dataset)
[plan.id for plan in matched_plans]

The same generated plan can be used through the public plan API by selecting the `auto` store.

In [ ]:
plan_id = "woodpecker.normalize_tas_units_to_kelvin"

findings = woodpecker.plan.check(dataset, None, store_type="auto", plan_id=plan_id)
findings.fix_ids

In [ ]:
write = woodpecker.plan.fix(dataset, None, store_type="auto", plan_id=plan_id, write=True)

(
    write.stats,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values + 273.15),
)